In [1]:
import pandas as pd
import re

In [2]:
def calculate_intern_use_freq(df: pd.DataFrame) -> pd.DataFrame:
    df["Taxa de Uso Interna (%)"] = (df["Freq Total"] / df["Freq Total"].sum()) * 100

    return df

In [3]:
def read_rule_freq(path: str):
    rows = list()
    pattern = re.compile(r"^(.*?)\s+(\d+)\s+(\d+)\s+([\d.]+)$")

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            match = pattern.match(line)
            if match:
                regra, freq_total, arquivos, media_sent = match.groups()
                rows.append({
                    "Regra": regra.strip(),
                    "Freq Total": int(freq_total),
                    "Arquivos": int(arquivos),
                    "Média/Sent": float(media_sent)
                })
    df = pd.DataFrame(rows)
    df["Taxa de Uso (%)"] = (df["Freq Total"] / df["Freq Total"].sum()) * 100

    return df

In [4]:
def get_not_root_rules(df: pd.DataFrame) -> pd.DataFrame:
    df_not_root = df[~df["Regra"].str.startswith("*")].copy()

    return calculate_intern_use_freq(df=df_not_root)

In [5]:
def get_root_rules(df: pd.DataFrame) -> pd.DataFrame:
    df_root = df[df["Regra"].str.startswith("*")].copy()

    return calculate_intern_use_freq(df=df_root)

In [6]:
def get_exclusive_rules(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    df = df1[~df1["Regra"].isin(df2["Regra"])].copy()

    return calculate_intern_use_freq(df=df)

In [7]:
def get_simple_rules(df: pd.DataFrame) -> pd.DataFrame:
    pattern = r'^[A-Z]+\(\s*\*\s*\)$'
    mask = (~df["Regra"].str.startswith("*", na=False)) & df["Regra"].str.match(pattern, na=False)
    df = df[mask].copy()
    
    return calculate_intern_use_freq(df=df)

In [8]:
def get_complex_rules(df: pd.DataFrame) -> pd.DataFrame:
    pattern = r'^[A-Z]+\(\s*\*\s*\)$'
    mask = (~df["Regra"].str.startswith("*", na=False)) & (~df["Regra"].str.match(pattern, na=False))
    df = df[mask].copy()
    
    return calculate_intern_use_freq(df=df)

### UPOS e DEPREL

In [9]:
llm_rules = read_rule_freq(path=r"aggregated_output/llm/rule_frequencies.txt")
human_rules = read_rule_freq(path=r"aggregated_output/human/rule_frequencies.txt")

#### 10 Regras Mais Utilizadas pela Máquina

In [10]:
llm_rules.head(10)

,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%)
0,NOUN(*),404676,5391,5.1115,13.772477
1,ADP(*),311263,5391,3.9316,10.593320
2,DET(*),310448,5391,3.9213,10.565583
3,PUNCT(*),271593,5391,3.4305,9.243218
4,PROPN(*),188795,5385,2.3847,6.425325
5,VERB(*),145206,5390,1.8341,4.941846
6,ADJ(*),139581,5389,1.7631,4.750408
7,*(VERB),68865,5389,0.8698,2.343706
8,ADV(*),67749,5383,0.8557,2.305725
9,CCONJ(*),53107,5383,0.6708,1.807409


#### 10 Regras Mais Utilizadas pelo Humano

In [11]:
human_rules.head(10)

,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%)
0,NOUN(*),218130,5388,4.3242,13.670248
1,ADP(*),167825,5389,3.3270,10.517624
2,DET(*),166886,5383,3.3083,10.458777
3,PUNCT(*),148522,5390,2.9443,9.307902
4,VERB(*),83570,5367,1.6567,5.237348
5,PROPN(*),82755,4637,1.6405,5.186271
6,ADJ(*),48915,5246,0.9697,3.065512
7,ADV(*),41476,5174,0.8222,2.599309
8,PRON(*),41110,5083,0.8150,2.576371
9,*(VERB),39713,5372,0.7873,2.488821


#### 10 Regras Menos Utilizadas pela Máquina

In [12]:
llm_rules.tail(10)

,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%)
57648,"VERB(PRON/obj, *, PROPN/obl, NOUN/obl)",1,1,0.0,0.000034
57649,"VERB(CCONJ/cc, PRON/nsubj:outer, AUX/cop, SCON...",1,1,0.0,0.000034
57650,"VERB(SCONJ/mark, NOUN/obl, *, PROPN/obl, NOUN/...",1,1,0.0,0.000034
57651,"VERB(PUNCT/punct, NOUN/nsubj, ADV/advmod, AUX/...",1,1,0.0,0.000034
57652,"VERB(SCONJ/mark, PROPN/nsubj:pass, AUX/aux:pas...",1,1,0.0,0.000034
57653,"VERB(PUNCT/punct, SCONJ/mark, NOUN/nsubj, *, N...",1,1,0.0,0.000034
57654,"VERB(SCONJ/mark, PRON/nsubj, *, NOUN/obl, ADJ/...",1,1,0.0,0.000034
57655,"ADJ(PUNCT/punct, CCONJ/cc, SCONJ/mark, NOUN/ns...",1,1,0.0,0.000034
57656,"PROPN(ADP/case, *, PROPN/appos, NOUN/conj)",1,1,0.0,0.000034
57657,"VERB(PROPN/nsubj, AUX/cop, ADV/advmod, *, NOUN...",1,1,0.0,0.000034


In [13]:
pd.set_option('display.max_colwidth', None)

#### 10 Regras Menos Utilizadas pelo Humano

In [14]:
human_rules.tail(10)

,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%)
60167,"PRON(CCONJ/cc, PUNCT/punct, AUX/cop, *, DET/det, PUNCT/punct, PUNCT/punct)",1,1,0.0,0.000063
60168,"VERB(PUNCT/punct, ADV/advmod, *, PUNCT/punct, PRON/conj, VERB/conj, PUNCT/punct, VERB/conj)",1,1,0.0,0.000063
60169,"PRON(CCONJ/cc, PUNCT/punct, *, VERB/acl:relcl, PUNCT/punct)",1,1,0.0,0.000063
60170,"VERB(NUM/nsubj, VERB/ccomp:speech, PROPN/nsubj, *, VERB/ccomp, PUNCT/punct)",1,1,0.0,0.000063
60171,"VERB(PUNCT/punct, PUNCT/punct, *, ADJ/ccomp, PUNCT/punct, VERB/conj, PUNCT/punct)",1,1,0.0,0.000063
60172,"VERB(CCONJ/cc, PUNCT/punct, PRON/nsubj, *, NOUN/obj, PRON/obl)",1,1,0.0,0.000063
60173,"NOUN(PUNCT/punct, CCONJ/cc, PUNCT/punct, ADJ/amod, *, PUNCT/punct, NUM/punct, PRON/nsubj, ADV/advmod)",1,1,0.0,0.000063
60174,"VERB(NOUN/discourse, VERB/ccomp:speech, NOUN/nsubj, *, VERB/xcomp, PRON/parataxis, PUNCT/punct)",1,1,0.0,0.000063
60175,"VERB(ADP/mark, PUNCT/punct, SCONJ/mark, PRON/nsubj, ADV/advmod, *, NOUN/obl, PUNCT/punct, VERB/conj)",1,1,0.0,0.000063
60176,"VERB(CCONJ/cc, PUNCT/punct, PRON/nsubj, *, PRON/obj, NOUN/obl, VERB/conj, PUNCT/punct)",1,1,0.0,0.000063


#### 10 Regras Mais Utilizadas pelo LLM Sem Ser Raíz

In [15]:
llm_rules_not_root = get_not_root_rules(df=llm_rules)
llm_rules_not_root.head(10)

,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
0,NOUN(*),404676,5391,5.1115,13.772477,14.153841
1,ADP(*),311263,5391,3.9316,10.593320,10.886652
2,DET(*),310448,5391,3.9213,10.565583,10.858147
3,PUNCT(*),271593,5391,3.4305,9.243218,9.499165
4,PROPN(*),188795,5385,2.3847,6.425325,6.603244
5,VERB(*),145206,5390,1.8341,4.941846,5.078687
6,ADJ(*),139581,5389,1.7631,4.750408,4.881948
8,ADV(*),67749,5383,0.8557,2.305725,2.369571
9,CCONJ(*),53107,5383,0.6708,1.807409,1.857456
10,AUX(*),52834,5385,0.6673,1.798118,1.847908


#### 10 Regras Menos Utilizadas pelo LLM Sem Ser Raíz

In [16]:
llm_rules_not_root.tail(10)

,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
57648,"VERB(PRON/obj, *, PROPN/obl, NOUN/obl)",1,1,0.0,0.000034,0.000035
57649,"VERB(CCONJ/cc, PRON/nsubj:outer, AUX/cop, SCONJ/mark, ADV/advmod, PRON/nsubj, *, PRON/obj, NOUN/obl, VERB/advcl, PUNCT/punct)",1,1,0.0,0.000034,0.000035
57650,"VERB(SCONJ/mark, NOUN/obl, *, PROPN/obl, NOUN/obl, VERB/advcl)",1,1,0.0,0.000034,0.000035
57651,"VERB(PUNCT/punct, NOUN/nsubj, ADV/advmod, AUX/cop, *, NOUN/obj, VERB/conj, VERB/advcl, PUNCT/punct, PUNCT/punct)",1,1,0.0,0.000034,0.000035
57652,"VERB(SCONJ/mark, PROPN/nsubj:pass, AUX/aux:pass, *, PROPN/obl, VERB/advcl)",1,1,0.0,0.000034,0.000035
57653,"VERB(PUNCT/punct, SCONJ/mark, NOUN/nsubj, *, NOUN/ccomp, VERB/advcl)",1,1,0.0,0.000034,0.000035
57654,"VERB(SCONJ/mark, PRON/nsubj, *, NOUN/obl, ADJ/conj)",1,1,0.0,0.000034,0.000035
57655,"ADJ(PUNCT/punct, CCONJ/cc, SCONJ/mark, NOUN/nsubj, ADV/advmod, AUX/cop, *, VERB/advcl)",1,1,0.0,0.000034,0.000035
57656,"PROPN(ADP/case, *, PROPN/appos, NOUN/conj)",1,1,0.0,0.000034,0.000035
57657,"VERB(PROPN/nsubj, AUX/cop, ADV/advmod, *, NOUN/obj, PUNCT/punct)",1,1,0.0,0.000034,0.000035


#### 10 Regras Mais Utilizadas pelo Humano Sem Ser Raíz

In [17]:
human_rules_not_root = get_not_root_rules(df=human_rules)
human_rules_not_root.head(10)

,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
0,NOUN(*),218130,5388,4.3242,13.670248,14.116519
1,ADP(*),167825,5389,3.3270,10.517624,10.860976
2,DET(*),166886,5383,3.3083,10.458777,10.800208
3,PUNCT(*),148522,5390,2.9443,9.307902,9.611762
4,VERB(*),83570,5367,1.6567,5.237348,5.408323
5,PROPN(*),82755,4637,1.6405,5.186271,5.355579
6,ADJ(*),48915,5246,0.9697,3.065512,3.165587
7,ADV(*),41476,5174,0.8222,2.599309,2.684164
8,PRON(*),41110,5083,0.8150,2.576371,2.660478
10,AUX(*),34717,5157,0.6882,2.175721,2.246748


In [18]:
human_rules_not_root.tail(10)

,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
60167,"PRON(CCONJ/cc, PUNCT/punct, AUX/cop, *, DET/det, PUNCT/punct, PUNCT/punct)",1,1,0.0,0.000063,0.000065
60168,"VERB(PUNCT/punct, ADV/advmod, *, PUNCT/punct, PRON/conj, VERB/conj, PUNCT/punct, VERB/conj)",1,1,0.0,0.000063,0.000065
60169,"PRON(CCONJ/cc, PUNCT/punct, *, VERB/acl:relcl, PUNCT/punct)",1,1,0.0,0.000063,0.000065
60170,"VERB(NUM/nsubj, VERB/ccomp:speech, PROPN/nsubj, *, VERB/ccomp, PUNCT/punct)",1,1,0.0,0.000063,0.000065
60171,"VERB(PUNCT/punct, PUNCT/punct, *, ADJ/ccomp, PUNCT/punct, VERB/conj, PUNCT/punct)",1,1,0.0,0.000063,0.000065
60172,"VERB(CCONJ/cc, PUNCT/punct, PRON/nsubj, *, NOUN/obj, PRON/obl)",1,1,0.0,0.000063,0.000065
60173,"NOUN(PUNCT/punct, CCONJ/cc, PUNCT/punct, ADJ/amod, *, PUNCT/punct, NUM/punct, PRON/nsubj, ADV/advmod)",1,1,0.0,0.000063,0.000065
60174,"VERB(NOUN/discourse, VERB/ccomp:speech, NOUN/nsubj, *, VERB/xcomp, PRON/parataxis, PUNCT/punct)",1,1,0.0,0.000063,0.000065
60175,"VERB(ADP/mark, PUNCT/punct, SCONJ/mark, PRON/nsubj, ADV/advmod, *, NOUN/obl, PUNCT/punct, VERB/conj)",1,1,0.0,0.000063,0.000065
60176,"VERB(CCONJ/cc, PUNCT/punct, PRON/nsubj, *, PRON/obj, NOUN/obl, VERB/conj, PUNCT/punct)",1,1,0.0,0.000063,0.000065


#### Regras de Raiz Usadas pelo LLM

In [19]:
llm_rules_root = get_root_rules(df=llm_rules)
print(f">> Número de Regras de Raiz de LLM: {llm_rules_root.shape[0]}")
print(f">> Total de Frequências de Regras de Raiz Usadas por LLM: {llm_rules_root['Freq Total'].sum()}")
llm_rules_root

>> Número de Regras de Raiz de LLM: 15
>> Total de Frequências de Regras de Raiz Usadas por LLM: 79170


,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
7,*(VERB),68865,5389,0.8698,2.343706,86.983706
37,*(NOUN),5497,2793,0.0694,0.187081,6.943287
45,*(ADJ),3258,2295,0.0412,0.110881,4.115195
155,*(PROPN),680,428,0.0086,0.023143,0.858911
361,*(PRON),234,213,0.0030,0.007964,0.295567
422,*(NUM),196,156,0.0025,0.006671,0.247569
519,*(ADV),155,145,0.0020,0.005275,0.195781
539,*(PUNCT),148,99,0.0019,0.005037,0.186939
916,*(SYM),70,48,0.0009,0.002382,0.088417
1683,*(X),32,12,0.0004,0.001089,0.040419


#### Regras de Raiz Usadas pelo Humano

In [20]:
human_rules_root = get_root_rules(df=human_rules)
print(f">> Número de Regras de Raiz de Humano: {human_rules_root.shape[0]}")
print(f">> Total de Frequências de Regras de Raiz Usadas por Humano: {human_rules_root['Freq Total'].sum()}")
human_rules_root

>> Número de Regras de Raiz de Humano: 16
>> Total de Frequências de Regras de Raiz Usadas por Humano: 50444


,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
9,*(VERB),39713,5372,0.7873,2.488821,78.726905
21,*(NOUN),6049,3092,0.1199,0.379092,11.991515
39,*(ADJ),2310,1601,0.0458,0.144768,4.579336
87,*(PROPN),705,540,0.0140,0.044182,1.397589
103,*(PRON),599,492,0.0119,0.037539,1.187455
135,*(ADV),429,334,0.0085,0.026886,0.850448
198,*(NUM),254,200,0.0050,0.015918,0.503529
505,*(SYM),77,56,0.0015,0.004826,0.152645
519,*(X),75,64,0.0015,0.004700,0.148680
572,*(AUX),67,65,0.0013,0.004199,0.132821


#### Regras Exclusivas de LLM

In [21]:
exclusive_llm_rules = get_exclusive_rules(df1=llm_rules, df2=human_rules)
print(f">> Número de Regras Exclusivas de LLM: {exclusive_llm_rules.shape[0]}")
print(f">> Total de Frequências de Regras Exclusivas de LLM: {exclusive_llm_rules['Freq Total'].sum()}")
exclusive_llm_rules.head(10)

>> Número de Regras Exclusivas de LLM: 41092
>> Total de Frequências de Regras Exclusivas de LLM: 60028


,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
739,"VERB(PUNCT/punct, SCONJ/mark, PROPN/nsubj, *, NOUN/obj)",94,93,0.0012,0.003199,0.156594
884,"VERB(PUNCT/punct, ADV/advmod, *, NOUN/obl:agent, PUNCT/punct)",73,73,0.0009,0.002484,0.121610
958,"NOUN(ADV/advmod, AUX/cop, DET/det, ADJ/amod, *, VERB/acl, PUNCT/punct)",66,66,0.0008,0.002246,0.109949
996,"ADJ(PUNCT/punct, NOUN/nsubj, AUX/cop, *, VERB/conj, PUNCT/punct, PUNCT/punct)",62,60,0.0008,0.002110,0.103285
1124,"VERB(NOUN/nsubj, NOUN/obl, *, NOUN/obj, VERB/advcl, PUNCT/punct)",54,54,0.0007,0.001838,0.089958
1294,"VERB(ADJ/root, NOUN/nsubj, *, NOUN/obj, PUNCT/punct)",44,44,0.0006,0.001497,0.073299
1334,"VERB(ADP/cc, NOUN/nsubj, *, ADJ/ccomp, PUNCT/punct)",42,42,0.0005,0.001429,0.069967
1363,"ADV(PUNCT/punct, AUX/cop, *, NOUN/obl, PUNCT/punct, PUNCT/punct)",41,41,0.0005,0.001395,0.068301
1461,"NOUN(*, ADP/fixed)",38,38,0.0005,0.001293,0.063304
1488,"NOUN(PUNCT/punct, ADP/case, *, PROPN/nmod, VERB/acl)",37,37,0.0005,0.001259,0.061638


In [22]:
exclusive_llm_rules.tail(10)

,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
57646,"NOUN(CCONJ/cc, SCONJ/mark, NOUN/nsubj, AUX/cop, ADV/advmod, ADP/case, *)",1,1,0.0,0.000034,0.001666
57647,"VERB(NOUN/nsubj, *, PRON/obj, NOUN/xcomp, VERB/conj, PUNCT/punct)",1,1,0.0,0.000034,0.001666
57649,"VERB(CCONJ/cc, PRON/nsubj:outer, AUX/cop, SCONJ/mark, ADV/advmod, PRON/nsubj, *, PRON/obj, NOUN/obl, VERB/advcl, PUNCT/punct)",1,1,0.0,0.000034,0.001666
57650,"VERB(SCONJ/mark, NOUN/obl, *, PROPN/obl, NOUN/obl, VERB/advcl)",1,1,0.0,0.000034,0.001666
57651,"VERB(PUNCT/punct, NOUN/nsubj, ADV/advmod, AUX/cop, *, NOUN/obj, VERB/conj, VERB/advcl, PUNCT/punct, PUNCT/punct)",1,1,0.0,0.000034,0.001666
57652,"VERB(SCONJ/mark, PROPN/nsubj:pass, AUX/aux:pass, *, PROPN/obl, VERB/advcl)",1,1,0.0,0.000034,0.001666
57653,"VERB(PUNCT/punct, SCONJ/mark, NOUN/nsubj, *, NOUN/ccomp, VERB/advcl)",1,1,0.0,0.000034,0.001666
57654,"VERB(SCONJ/mark, PRON/nsubj, *, NOUN/obl, ADJ/conj)",1,1,0.0,0.000034,0.001666
57656,"PROPN(ADP/case, *, PROPN/appos, NOUN/conj)",1,1,0.0,0.000034,0.001666
57657,"VERB(PROPN/nsubj, AUX/cop, ADV/advmod, *, NOUN/obj, PUNCT/punct)",1,1,0.0,0.000034,0.001666


#### Regras Exclusivas de Humano

In [23]:
exclusive_human_rules = get_exclusive_rules(df1=human_rules, df2=llm_rules)
print(f">> Número de Regras Exclusivas de Humano: {exclusive_human_rules.shape[0]}")
print(f">> Total de Frequências de Regras Exclusivas de Humano: {exclusive_human_rules['Freq Total'].sum()}")
exclusive_human_rules.head(10)

>> Número de Regras Exclusivas de Humano: 43611
>> Total de Frequências de Regras Exclusivas de Humano: 54510


,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
302,"NOUN(PUNCT/punct, NOUN/nsubj, CCONJ/cc, DET/det, *, NOUN/nmod, PUNCT/punct)",152,152,0.0030,0.009526,0.278848
326,"VERB(VERB/advcl, *, NOUN/obl, NOUN/obl, NOUN/obl, PUNCT/punct)",138,138,0.0027,0.008648,0.253165
526,"CCONJ(*, AUX/fixed, PUNCT/punct)",74,74,0.0015,0.004638,0.135755
587,"PUNCT(PUNCT/punct, *, PUNCT/parataxis, PUNCT/punct)",65,61,0.0013,0.004074,0.119244
655,"NOUN(SYM/nmod, NUM/nummod, PUNCT/punct, NUM/nummod, *)",59,45,0.0012,0.003698,0.108237
755,"NUM(PUNCT/punct, *, NUM/flat)",50,28,0.0010,0.003134,0.091726
842,"X(*, X/flat:name)",43,38,0.0009,0.002695,0.078885
980,"NOUN(ADP/case, DET/det, *, X/conj, NOUN/conj)",36,36,0.0007,0.002256,0.066043
1117,"NOUN(PUNCT/punct, *, PUNCT/punct, SYM/parataxis, PUNCT/punct)",30,1,0.0006,0.001880,0.055036
1167,"NOUN(PUNCT/punct, *, NOUN/flat:name)",28,24,0.0006,0.001755,0.051367


In [24]:
exclusive_llm_rules.tail(10)

,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
57646,"NOUN(CCONJ/cc, SCONJ/mark, NOUN/nsubj, AUX/cop, ADV/advmod, ADP/case, *)",1,1,0.0,0.000034,0.001666
57647,"VERB(NOUN/nsubj, *, PRON/obj, NOUN/xcomp, VERB/conj, PUNCT/punct)",1,1,0.0,0.000034,0.001666
57649,"VERB(CCONJ/cc, PRON/nsubj:outer, AUX/cop, SCONJ/mark, ADV/advmod, PRON/nsubj, *, PRON/obj, NOUN/obl, VERB/advcl, PUNCT/punct)",1,1,0.0,0.000034,0.001666
57650,"VERB(SCONJ/mark, NOUN/obl, *, PROPN/obl, NOUN/obl, VERB/advcl)",1,1,0.0,0.000034,0.001666
57651,"VERB(PUNCT/punct, NOUN/nsubj, ADV/advmod, AUX/cop, *, NOUN/obj, VERB/conj, VERB/advcl, PUNCT/punct, PUNCT/punct)",1,1,0.0,0.000034,0.001666
57652,"VERB(SCONJ/mark, PROPN/nsubj:pass, AUX/aux:pass, *, PROPN/obl, VERB/advcl)",1,1,0.0,0.000034,0.001666
57653,"VERB(PUNCT/punct, SCONJ/mark, NOUN/nsubj, *, NOUN/ccomp, VERB/advcl)",1,1,0.0,0.000034,0.001666
57654,"VERB(SCONJ/mark, PRON/nsubj, *, NOUN/obl, ADJ/conj)",1,1,0.0,0.000034,0.001666
57656,"PROPN(ADP/case, *, PROPN/appos, NOUN/conj)",1,1,0.0,0.000034,0.001666
57657,"VERB(PROPN/nsubj, AUX/cop, ADV/advmod, *, NOUN/obj, PUNCT/punct)",1,1,0.0,0.000034,0.001666


#### Regras de Raiz Exclusivas de LLM

In [25]:
exclusive_llm_root_rules = get_exclusive_rules(df1=llm_rules_root, df2=human_rules_root)
print(f">> Número de Regras de Raiz Exclusivas de LLM: {exclusive_llm_root_rules.shape[0]}")
print(f">> Total de Frequências de Regras de Raiz Exclusivas de LLM: {exclusive_llm_root_rules['Freq Total'].sum()}")
exclusive_llm_root_rules.head(10)

>> Número de Regras de Raiz Exclusivas de LLM: 0
>> Total de Frequências de Regras de Raiz Exclusivas de LLM: 0


,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)


#### Regras de Raiz Exclusivas de Humano

In [26]:
exclusive_human_root_rules = get_exclusive_rules(df1=human_rules_root, df2=llm_rules_root)
print(f">> Número de Regras de Raiz Exclusivas de Humano: {exclusive_human_root_rules.shape[0]}")
print(f">> Total de Frequências de Regras de Raiz Exclusivas de Humano: {exclusive_human_root_rules['Freq Total'].sum()}")
exclusive_human_root_rules.head(10)

>> Número de Regras de Raiz Exclusivas de Humano: 1
>> Total de Frequências de Regras de Raiz Exclusivas de Humano: 5


,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
4881,*(CCONJ),5,5,0.0001,0.000313,100.0


#### Regras Simples de LLM

In [27]:
llm_simple_rules = get_simple_rules(df=llm_rules)
print(f">> Número de Regras Simples de LLM: {llm_simple_rules.shape[0]}")
print(f">> Total de Frequências de Regras Simples de LLM: {llm_simple_rules['Freq Total'].sum()}")
llm_simple_rules

>> Número de Regras Simples de LLM: 16
>> Total de Frequências de Regras Simples de LLM: 2058014


,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
0,NOUN(*),404676,5391,5.1115,13.772477,19.663423
1,ADP(*),311263,5391,3.9316,10.593320,15.124435
2,DET(*),310448,5391,3.9213,10.565583,15.084834
3,PUNCT(*),271593,5391,3.4305,9.243218,13.196849
4,PROPN(*),188795,5385,2.3847,6.425325,9.173650
5,VERB(*),145206,5390,1.8341,4.941846,7.055637
6,ADJ(*),139581,5389,1.7631,4.750408,6.782315
8,ADV(*),67749,5383,0.8557,2.305725,3.291960
9,CCONJ(*),53107,5383,0.6708,1.807409,2.580498
10,AUX(*),52834,5385,0.6673,1.798118,2.567232


#### Regras Simples de Humano

In [28]:
human_simple_rules = get_simple_rules(df=human_rules)
print(f">> Número de Regras Simples de Humano: {human_simple_rules.shape[0]}")
print(f">> Total de Frequências de Regras Simples de Humano: {human_simple_rules['Freq Total'].sum()}")
human_simple_rules

>> Número de Regras Simples de Humano: 16
>> Total de Frequências de Regras Simples de Humano: 1109365


,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
0,NOUN(*),218130,5388,4.3242,13.670248,19.662600
1,ADP(*),167825,5389,3.3270,10.517624,15.128024
2,DET(*),166886,5383,3.3083,10.458777,15.043381
3,PUNCT(*),148522,5390,2.9443,9.307902,13.388019
4,VERB(*),83570,5367,1.6567,5.237348,7.533138
5,PROPN(*),82755,4637,1.6405,5.186271,7.459673
6,ADJ(*),48915,5246,0.9697,3.065512,4.409279
7,ADV(*),41476,5174,0.8222,2.599309,3.738715
8,PRON(*),41110,5083,0.8150,2.576371,3.705724
10,AUX(*),34717,5157,0.6882,2.175721,3.129448


#### Regras Complexas de LLM

In [29]:
llm_complex_rules = get_complex_rules(df=llm_rules)
print(f">> Número de Regras Complexas de LLM: {llm_complex_rules.shape[0]}")
print(f">> Total de Frequências de Regras Complexas de LLM: {llm_complex_rules['Freq Total'].sum()}")
llm_complex_rules.head(10)

>> Número de Regras Complexas de LLM: 57627
>> Total de Frequências de Regras Complexas de LLM: 801111


,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
12,"NOUN(DET/det, *)",39829,5366,0.5031,1.355514,4.971721
13,"NOUN(ADP/case, DET/det, *)",34880,5332,0.4406,1.187083,4.353953
15,"NOUN(ADP/case, *)",29415,5278,0.3715,1.001091,3.671776
17,"PROPN(ADP/case, DET/det, *)",23601,5031,0.2981,0.803221,2.946034
18,"NOUN(DET/det, *, NOUN/nmod)",18003,5058,0.2274,0.612702,2.247254
19,"PROPN(ADP/case, *)",16102,4615,0.2034,0.548005,2.009959
20,"NOUN(ADP/case, DET/det, *, ADJ/amod)",12705,4693,0.1605,0.432394,1.585923
21,"NOUN(ADP/case, DET/det, *, NOUN/nmod)",12602,4613,0.1592,0.428888,1.573065
22,"NOUN(ADP/case, *, ADJ/amod)",11720,4467,0.1480,0.398871,1.462968
23,"NOUN(DET/det, *, ADJ/amod)",11550,4534,0.1459,0.393085,1.441748


In [30]:
llm_complex_rules.tail(10)

,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
57648,"VERB(PRON/obj, *, PROPN/obl, NOUN/obl)",1,1,0.0,0.000034,0.000125
57649,"VERB(CCONJ/cc, PRON/nsubj:outer, AUX/cop, SCONJ/mark, ADV/advmod, PRON/nsubj, *, PRON/obj, NOUN/obl, VERB/advcl, PUNCT/punct)",1,1,0.0,0.000034,0.000125
57650,"VERB(SCONJ/mark, NOUN/obl, *, PROPN/obl, NOUN/obl, VERB/advcl)",1,1,0.0,0.000034,0.000125
57651,"VERB(PUNCT/punct, NOUN/nsubj, ADV/advmod, AUX/cop, *, NOUN/obj, VERB/conj, VERB/advcl, PUNCT/punct, PUNCT/punct)",1,1,0.0,0.000034,0.000125
57652,"VERB(SCONJ/mark, PROPN/nsubj:pass, AUX/aux:pass, *, PROPN/obl, VERB/advcl)",1,1,0.0,0.000034,0.000125
57653,"VERB(PUNCT/punct, SCONJ/mark, NOUN/nsubj, *, NOUN/ccomp, VERB/advcl)",1,1,0.0,0.000034,0.000125
57654,"VERB(SCONJ/mark, PRON/nsubj, *, NOUN/obl, ADJ/conj)",1,1,0.0,0.000034,0.000125
57655,"ADJ(PUNCT/punct, CCONJ/cc, SCONJ/mark, NOUN/nsubj, ADV/advmod, AUX/cop, *, VERB/advcl)",1,1,0.0,0.000034,0.000125
57656,"PROPN(ADP/case, *, PROPN/appos, NOUN/conj)",1,1,0.0,0.000034,0.000125
57657,"VERB(PROPN/nsubj, AUX/cop, ADV/advmod, *, NOUN/obj, PUNCT/punct)",1,1,0.0,0.000034,0.000125


#### Regras Complexas de Humano

In [31]:
human_complex_rules = get_complex_rules(df=human_rules)
print(f">> Número de Regras Complexas de Humano: {human_complex_rules.shape[0]}")
print(f">> Total de Frequências de Regras Complexas de Humano: {human_complex_rules['Freq Total'].sum()}")
human_complex_rules.head(10)

>> Número de Regras Complexas de Humano: 60145
>> Total de Frequências de Regras Complexas de Humano: 435846


,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
12,"NOUN(ADP/case, DET/det, *)",25249,5066,0.5005,1.582360,5.793101
13,"NOUN(DET/det, *)",22984,4930,0.4556,1.440412,5.273422
16,"NOUN(ADP/case, *)",16490,4557,0.3269,1.033431,3.783446
17,"PROPN(ADP/case, DET/det, *)",11473,3144,0.2274,0.719015,2.632352
18,"NOUN(DET/det, *, NOUN/nmod)",8297,3733,0.1645,0.519975,1.903654
19,"NOUN(ADP/case, DET/det, *, NOUN/nmod)",7981,3609,0.1582,0.500171,1.831151
20,"PROPN(ADP/case, *)",6057,2610,0.1201,0.379593,1.389711
22,"PROPN(*, PROPN/flat:name)",5824,2725,0.1155,0.364991,1.336252
23,"NOUN(ADP/case, *, NOUN/nmod)",4995,2791,0.0990,0.313038,1.146047
24,"NOUN(*, NOUN/nmod)",4836,2812,0.0959,0.303073,1.109566


In [32]:
human_complex_rules.tail(10)

,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
60167,"PRON(CCONJ/cc, PUNCT/punct, AUX/cop, *, DET/det, PUNCT/punct, PUNCT/punct)",1,1,0.0,0.000063,0.000229
60168,"VERB(PUNCT/punct, ADV/advmod, *, PUNCT/punct, PRON/conj, VERB/conj, PUNCT/punct, VERB/conj)",1,1,0.0,0.000063,0.000229
60169,"PRON(CCONJ/cc, PUNCT/punct, *, VERB/acl:relcl, PUNCT/punct)",1,1,0.0,0.000063,0.000229
60170,"VERB(NUM/nsubj, VERB/ccomp:speech, PROPN/nsubj, *, VERB/ccomp, PUNCT/punct)",1,1,0.0,0.000063,0.000229
60171,"VERB(PUNCT/punct, PUNCT/punct, *, ADJ/ccomp, PUNCT/punct, VERB/conj, PUNCT/punct)",1,1,0.0,0.000063,0.000229
60172,"VERB(CCONJ/cc, PUNCT/punct, PRON/nsubj, *, NOUN/obj, PRON/obl)",1,1,0.0,0.000063,0.000229
60173,"NOUN(PUNCT/punct, CCONJ/cc, PUNCT/punct, ADJ/amod, *, PUNCT/punct, NUM/punct, PRON/nsubj, ADV/advmod)",1,1,0.0,0.000063,0.000229
60174,"VERB(NOUN/discourse, VERB/ccomp:speech, NOUN/nsubj, *, VERB/xcomp, PRON/parataxis, PUNCT/punct)",1,1,0.0,0.000063,0.000229
60175,"VERB(ADP/mark, PUNCT/punct, SCONJ/mark, PRON/nsubj, ADV/advmod, *, NOUN/obl, PUNCT/punct, VERB/conj)",1,1,0.0,0.000063,0.000229
60176,"VERB(CCONJ/cc, PUNCT/punct, PRON/nsubj, *, PRON/obj, NOUN/obl, VERB/conj, PUNCT/punct)",1,1,0.0,0.000063,0.000229


#### Regras Complexas Exclusivas de LLM

In [33]:
llm_complex_exclusive_rules = get_exclusive_rules(df1=llm_complex_rules, df2=human_complex_rules)
print(f">> Número de Regras Complexas Exclusivas de LLM: {llm_complex_exclusive_rules.shape[0]}")
print(f">> Total de Frequências de Regras Complexas Exclusivas de LLM: {llm_complex_exclusive_rules['Freq Total'].sum()}")
llm_complex_exclusive_rules.head(10)

>> Número de Regras Complexas Exclusivas de LLM: 41092
>> Total de Frequências de Regras Complexas Exclusivas de LLM: 60028


,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
739,"VERB(PUNCT/punct, SCONJ/mark, PROPN/nsubj, *, NOUN/obj)",94,93,0.0012,0.003199,0.156594
884,"VERB(PUNCT/punct, ADV/advmod, *, NOUN/obl:agent, PUNCT/punct)",73,73,0.0009,0.002484,0.121610
958,"NOUN(ADV/advmod, AUX/cop, DET/det, ADJ/amod, *, VERB/acl, PUNCT/punct)",66,66,0.0008,0.002246,0.109949
996,"ADJ(PUNCT/punct, NOUN/nsubj, AUX/cop, *, VERB/conj, PUNCT/punct, PUNCT/punct)",62,60,0.0008,0.002110,0.103285
1124,"VERB(NOUN/nsubj, NOUN/obl, *, NOUN/obj, VERB/advcl, PUNCT/punct)",54,54,0.0007,0.001838,0.089958
1294,"VERB(ADJ/root, NOUN/nsubj, *, NOUN/obj, PUNCT/punct)",44,44,0.0006,0.001497,0.073299
1334,"VERB(ADP/cc, NOUN/nsubj, *, ADJ/ccomp, PUNCT/punct)",42,42,0.0005,0.001429,0.069967
1363,"ADV(PUNCT/punct, AUX/cop, *, NOUN/obl, PUNCT/punct, PUNCT/punct)",41,41,0.0005,0.001395,0.068301
1461,"NOUN(*, ADP/fixed)",38,38,0.0005,0.001293,0.063304
1488,"NOUN(PUNCT/punct, ADP/case, *, PROPN/nmod, VERB/acl)",37,37,0.0005,0.001259,0.061638


In [34]:
llm_complex_exclusive_rules.tail(10)

,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
57646,"NOUN(CCONJ/cc, SCONJ/mark, NOUN/nsubj, AUX/cop, ADV/advmod, ADP/case, *)",1,1,0.0,0.000034,0.001666
57647,"VERB(NOUN/nsubj, *, PRON/obj, NOUN/xcomp, VERB/conj, PUNCT/punct)",1,1,0.0,0.000034,0.001666
57649,"VERB(CCONJ/cc, PRON/nsubj:outer, AUX/cop, SCONJ/mark, ADV/advmod, PRON/nsubj, *, PRON/obj, NOUN/obl, VERB/advcl, PUNCT/punct)",1,1,0.0,0.000034,0.001666
57650,"VERB(SCONJ/mark, NOUN/obl, *, PROPN/obl, NOUN/obl, VERB/advcl)",1,1,0.0,0.000034,0.001666
57651,"VERB(PUNCT/punct, NOUN/nsubj, ADV/advmod, AUX/cop, *, NOUN/obj, VERB/conj, VERB/advcl, PUNCT/punct, PUNCT/punct)",1,1,0.0,0.000034,0.001666
57652,"VERB(SCONJ/mark, PROPN/nsubj:pass, AUX/aux:pass, *, PROPN/obl, VERB/advcl)",1,1,0.0,0.000034,0.001666
57653,"VERB(PUNCT/punct, SCONJ/mark, NOUN/nsubj, *, NOUN/ccomp, VERB/advcl)",1,1,0.0,0.000034,0.001666
57654,"VERB(SCONJ/mark, PRON/nsubj, *, NOUN/obl, ADJ/conj)",1,1,0.0,0.000034,0.001666
57656,"PROPN(ADP/case, *, PROPN/appos, NOUN/conj)",1,1,0.0,0.000034,0.001666
57657,"VERB(PROPN/nsubj, AUX/cop, ADV/advmod, *, NOUN/obj, PUNCT/punct)",1,1,0.0,0.000034,0.001666


#### Regras Complexas Exclusivas de Humano

In [35]:
human_complex_exclusive_rules = get_exclusive_rules(df1=human_complex_rules, df2=llm_complex_rules)
print(f">> Número de Regras Complexas Exclusivas de Humano: {human_complex_exclusive_rules.shape[0]}")
print(f">> Total de Frequências de Regras Complexas Exclusivas de Humano: {human_complex_exclusive_rules['Freq Total'].sum()}")
human_complex_exclusive_rules.head(10)

>> Número de Regras Complexas Exclusivas de Humano: 43610
>> Total de Frequências de Regras Complexas Exclusivas de Humano: 54505


,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
302,"NOUN(PUNCT/punct, NOUN/nsubj, CCONJ/cc, DET/det, *, NOUN/nmod, PUNCT/punct)",152,152,0.0030,0.009526,0.278873
326,"VERB(VERB/advcl, *, NOUN/obl, NOUN/obl, NOUN/obl, PUNCT/punct)",138,138,0.0027,0.008648,0.253188
526,"CCONJ(*, AUX/fixed, PUNCT/punct)",74,74,0.0015,0.004638,0.135767
587,"PUNCT(PUNCT/punct, *, PUNCT/parataxis, PUNCT/punct)",65,61,0.0013,0.004074,0.119255
655,"NOUN(SYM/nmod, NUM/nummod, PUNCT/punct, NUM/nummod, *)",59,45,0.0012,0.003698,0.108247
755,"NUM(PUNCT/punct, *, NUM/flat)",50,28,0.0010,0.003134,0.091735
842,"X(*, X/flat:name)",43,38,0.0009,0.002695,0.078892
980,"NOUN(ADP/case, DET/det, *, X/conj, NOUN/conj)",36,36,0.0007,0.002256,0.066049
1117,"NOUN(PUNCT/punct, *, PUNCT/punct, SYM/parataxis, PUNCT/punct)",30,1,0.0006,0.001880,0.055041
1167,"NOUN(PUNCT/punct, *, NOUN/flat:name)",28,24,0.0006,0.001755,0.051371


In [36]:
human_complex_exclusive_rules.tail(10)

,Regra,Freq Total,Arquivos,Média/Sent,Taxa de Uso (%),Taxa de Uso Interna (%)
60167,"PRON(CCONJ/cc, PUNCT/punct, AUX/cop, *, DET/det, PUNCT/punct, PUNCT/punct)",1,1,0.0,0.000063,0.001835
60168,"VERB(PUNCT/punct, ADV/advmod, *, PUNCT/punct, PRON/conj, VERB/conj, PUNCT/punct, VERB/conj)",1,1,0.0,0.000063,0.001835
60169,"PRON(CCONJ/cc, PUNCT/punct, *, VERB/acl:relcl, PUNCT/punct)",1,1,0.0,0.000063,0.001835
60170,"VERB(NUM/nsubj, VERB/ccomp:speech, PROPN/nsubj, *, VERB/ccomp, PUNCT/punct)",1,1,0.0,0.000063,0.001835
60171,"VERB(PUNCT/punct, PUNCT/punct, *, ADJ/ccomp, PUNCT/punct, VERB/conj, PUNCT/punct)",1,1,0.0,0.000063,0.001835
60172,"VERB(CCONJ/cc, PUNCT/punct, PRON/nsubj, *, NOUN/obj, PRON/obl)",1,1,0.0,0.000063,0.001835
60173,"NOUN(PUNCT/punct, CCONJ/cc, PUNCT/punct, ADJ/amod, *, PUNCT/punct, NUM/punct, PRON/nsubj, ADV/advmod)",1,1,0.0,0.000063,0.001835
60174,"VERB(NOUN/discourse, VERB/ccomp:speech, NOUN/nsubj, *, VERB/xcomp, PRON/parataxis, PUNCT/punct)",1,1,0.0,0.000063,0.001835
60175,"VERB(ADP/mark, PUNCT/punct, SCONJ/mark, PRON/nsubj, ADV/advmod, *, NOUN/obl, PUNCT/punct, VERB/conj)",1,1,0.0,0.000063,0.001835
60176,"VERB(CCONJ/cc, PUNCT/punct, PRON/nsubj, *, PRON/obj, NOUN/obl, VERB/conj, PUNCT/punct)",1,1,0.0,0.000063,0.001835
